# Lab 6 - Techniques of Optimization for training

## 1. Bias and Variance Trade-off

We will use the Twitter dataset again for this lab. The dataset contains tweets labeled as positive or negative sentiment.

In [1]:
import opendatasets as od
import pandas as pd
pd.set_option('display.max_colwidth', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
import re
import gensim.downloader
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout, Embedding, Flatten, LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from transformers import BertTokenizer, TFBertModel
from sklearn.model_selection import train_test_split
from scipy.special import softmax

# uv run python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")
nltk.download('vader_lexicon')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

2026-01-20 17:03:34.050636: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-20 17:03:34.137153: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-20 17:03:35.937650: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/javier/Documents/IA/IA learning/lab6/.venv/lib/python3.13/site-packages/keras/src/exp

True

In [4]:
columns = ['id', 'category', 'label', 'text']
training_data = pd.read_csv('../lab5/datasets/twitter-sentiment-analysis/twitter_training.csv', names=columns)
validation_data = pd.read_csv('../lab5/datasets/twitter-sentiment-analysis/twitter_validation.csv', names=columns)

training_data.sample(5)


,id,category,label,text
6394,300,Amazon,Negative,"That I like the familiar idea behind saying this and look whah it stands for, but don ’ t see this working"
7292,9254,Overwatch,Positive,"Friend wardens, roving"
30072,770,ApexLegends,Neutral,Positioning is everything
56132,11234,TomClancysRainbowSix,Positive,Guess the console wasn't ready for upgrade.. at least the villa's sunset looks beautiful... @ Ubisoft @ Rainbow6Game pic.fm / 8afBqD31g0
46418,11965,Verizon,Negative,@ Verizon YOURE THRODG OUR SPEEDS????? SHAME. ON.YOU! Home schooling and 2 people working from home and YOU are our only option. Thanks for NOTHING.


We will use the FFNN model from the previous lab as our base model. And we will improve it using different optimization techniques.